In [3]:
import spacy
import requests
import json
from tqdm import tqdm
import time
import re

nlp = spacy.load("en_core_web_sm")

### ConceptNet API Extraction

In [44]:
def get_conceptnet(word, rel=None, pos='start', limit=100):
    if rel is None:
        obj = requests.get(f"https://api.conceptnet.io/query?{pos}=/c/en/{word}&limit={limit}").json()
    else:
        obj = requests.get(f"https://api.conceptnet.io/query?{pos}=/c/en/{word}&rel=/r/{rel}&limit={limit}").json()
    return obj


In [ ]:
def get_verbs(word, weight=1):
    verb_rel = ["UsedFor", "ReceivesAction", "CapableOf"]
    verb_set = {}
    for v in verb_rel:
        obj = get_conceptnet(word, v)
        for edge in obj['edges']:
            # skip non-English edges
            if 'language' not in edge['end'] or edge['end']['language'] != 'en':
                continue

            word = edge['end']['label']
            if word in verb_set:
                verb_set[word] += edge['weight']
            else:
                verb_set[word] = edge['weight']


    # return only unique verbs using spacy to lemmatize
    verb_words = list(verb_set.keys())
    verb_graph = {}
    for w in verb_words:
        doc = nlp(w)
        for token in doc:
            if token.pos_ != "VERB":
                continue
            if token.lemma_ in verb_graph:
                verb_graph[token.lemma_] += verb_set[w]
            else:
                verb_graph[token.lemma_] = verb_set[w]

    # filter verbs by weight
    verb_graph = {k: round(v,2) for k, v in verb_graph.items() if v >= weight}

    return verb_graph


v = get_verbs("book",1)
print(v)
print(len(v))


{'learn': 43.25, 'study': 2.0, 'develop': 3.0, 'find': 1.0, 'publish': 2.0, 'write': 2.0, 'keep': 1.0, 'show': 1.0, 'paint': 1.0, 'resolve': 1.0, 'gather': 2.0, 'squash': 1.0, 'record': 3.0, 'build': 2.0, 'bind': 2.0, 'help': 2.0, 'hold': 3.0, 'read': 8.0, 'report': 1.0, 'absorb': 1.0, 'look': 6.0, 'emphasize': 1.0, 'organize': 1.0, 'gamble': 1.0, 'give': 1.0, 'wait': 1.0, 'journalize': 1.0, 'facilitate': 1.0, 'happen': 1.0, 'fill': 1.0, 'know': 1.0, 'prove': 1.0, 'communicate': 1.0, 'apply': 1.0, 'raise': 1.0, 'print': 1.0, 'pass': 1.0, 'solve': 1.0, 'explain': 1.0}
39


In [ ]:
def get_nouns(word, weight=1):
    noun_rel = ["RelatedTo", "PartOf", "IsA", "HasA", "MadeOf", "Synonym", "Antonym"]
    noun_set = {}
    for r in noun_rel:
        obj = get_conceptnet(word, r)
        for edge in obj['edges']:
            # skip non-English edges
            if 'language' in edge['end'] and edge['end']['language'] != 'en':
                continue

            word = edge['end']['label']
            if word in noun_set:
                noun_set[word] += edge['weight']
            else:
                noun_set[word] = edge['weight']
    
    noun_words = list(noun_set.keys())
    noun_graph = {}
    for w in noun_words:
        doc = nlp(w)
        for token in doc:
            if token.pos_ != "NOUN":
                continue
            if token.lemma_ in noun_graph:
                noun_graph[token.lemma_] += noun_set[w]
            else:
                noun_graph[token.lemma_] = noun_set[w]

    # filter nouns by weight
    noun_graph = {k: round(v,2) for k, v in noun_graph.items() if v >= weight}
    return noun_graph

n = get_nouns("book",1)
print(n)
print(len(n))

{'page': 12.54, 'library': 6.16, 'material': 6.8, 'novel': 4.29, 'paper': 4.02, 'chapter': 2.5, 'story': 2.93, 'tome': 3.57, 'literature': 1.42, 'thing': 1.25, 'volume': 2.5, 'text': 1.14, 'magazine': 1.07, 'spine': 1.03, 'grade': 1.0, 'textbook': 1.0, 'kind': 1.0, 'caution': 1.0, 'publication': 1.46, 'university': 1.0, 'reserve': 1.0, 'representation': 1.0, 'arm': 1.0, 'datum': 1.0, 'card': 2.0, 'book': 1.0, 'scroll': 1.0, 'trick': 1.0, 'container': 1.2, 'item': 1.38}
30


In [46]:
def get_relationship(word, weight=1):
    verb_rel = ["UsedFor", "ReceivesAction", "CapableOf"]
    rel_set = {}

    for v in verb_rel:
        obj = get_conceptnet(word, v, pos='start')
        for edge in obj['edges']:
            # skip non-English edges
            if 'language' not in edge['end'] or edge['end']['language'] != 'en':
                continue

            relation = edge['end']['label']
            if relation in rel_set:
                rel_set[relation] += edge['weight']
            else:
                rel_set[relation] = edge['weight']

    # break up into verbs and nouns
    all_words = list(rel_set.keys())
    rel_graph = {}      # format: '(verb, noun)': weight
    for w in all_words:
        doc = nlp(w)
        verb = None
        noun = None
        for token in doc:
            
            if token.pos_ == "VERB":
                verb = token.lemma_
            elif token.pos_ == "NOUN" and len(token.lemma_) > 2:
                noun = token.lemma_
        if verb and noun:
            rel_graph[(verb, noun)] = rel_set[w]
    
    # filter relationships by weight
    rel_graph = {k: round(v,2) for k, v in rel_graph.items() if v >= weight}
    return rel_graph


r = get_relationship("apple",1)
print(r)
print(len(r))

{('make', 'pie'): 1.0, ('get', 'carb'): 1.0, ('get', 'teacher'): 1.0, ('represent', 'knowlege'): 1.0, ('enjoy', 'fruit'): 1.0, ('make', 'applesauce'): 1.0, ('illustrate', 'gravity'): 1.0, ('grow', 'tree'): 1.0, ('pick', 'tree'): 1.0, ('test', 'gravity'): 1.0, ('illustrate', 'letter'): 1.0, ('keep', 'doctor'): 1.0, ('make', 'cider'): 1.0, ('fall', 'tree'): 1.0}
14


### Graph Building

In [68]:
# get list of nouns
NOUN_FILE = "bank_files/chatgpt_nouns.txt"
ALL_NOUNS = []
with open(NOUN_FILE, "r") as f:
    for line in f:
        ALL_NOUNS.append(line.strip())

In [69]:
# build full graph
FULL_WORD_GRAPH = {}
AUGMENT_NOUNS = []

with tqdm(total=len(ALL_NOUNS)) as pbar:
    for noun in ALL_NOUNS:
        pbar.set_description(f"Processing {noun}")

        rel_graph = get_relationship(noun,1)
        if rel_graph:
            # find missing nouns and add to augment list
            for k in rel_graph.keys():
                if k[1] not in ALL_NOUNS:
                    AUGMENT_NOUNS.append(k[1])
            FULL_WORD_GRAPH[noun] = rel_graph

        time.sleep(0.5)

        pbar.update(1)



Processing clock: 100%|██████████| 216/216 [02:36<00:00,  1.38it/s]          


In [ ]:
# add augmented nouns to graph
AUGMENT_NOUNS = list(set(AUGMENT_NOUNS))
print(f"Augmented nouns: {len(AUGMENT_NOUNS)}")

with tqdm(total=len(AUGMENT_NOUNS)) as pbar:
    for noun in AUGMENT_NOUNS:
        pbar.set_description(f"Processing {noun}")
        try:
            rel_graph = get_relationship(noun,1)
        except:
            rel_graph = None
        if rel_graph:
            # filter out relationships without matching nouns from the set of 1000 nouns
            FULL_WORD_GRAPH[noun] = rel_graph

        time.sleep(0.5)

        pbar.update(1)

Augmented nouns: 709


Processing cub:   8%|▊         | 56/709 [00:30<05:59,  1.82it/s]         


In [73]:
print(len(FULL_WORD_GRAPH))
print(FULL_WORD_GRAPH)

502
{'hammer': {('drive', 'nail'): 2.83, ('pound', 'nail'): 5.66, ('hit', 'nail'): 1.0, ('break', 'thing'): 1.0, ('pull', 'nail'): 1.0, ('break', 'fastener'): 1.0, ('smash', 'skull'): 1.0, ('kill', 'mouse'): 1.0, ('put', 'nail'): 1.0, ('knock', 'thing'): 1.0, ('knock', 'place'): 1.0, ('strike', 'gun'): 1.0, ('put', 'shoe'): 1.0, ('create', 'music'): 1.0, ('hit', 'string'): 1.0, ('straighten', 'nail'): 1.0, ('dig', 'hole'): 1.0, ('ring', 'bell'): 1.0, ('extend', 'arm'): 1.0, ('repair', 'dent'): 1.0, ('put', 'wood'): 1.0, ('flatten', 'anvil'): 1.0, ('break', 'object'): 1.0, ('force', 'wood'): 1.0, ('find', 'store'): 1.0, ('break', 'window'): 2.0, ('nail', 'nail'): 2.0, ('break', 'glass'): 1.0, ('force', 'board'): 1.0, ('strike', 'force'): 1.0, ('break', 'wall'): 1.0, ('nail', 'board'): 1.0}, 'samurai': {('fight', 'ninja'): 1.0}, 'wolf': {('smell', 'human'): 1.0, ('give', 'howl'): 1.0, ('kill', 'deer'): 1.0, ('mark', 'territory'): 1.0}, 'milk': {('make', 'pudding'): 1.0, ('feed', 'baby'):

In [ ]:
# save to json (separated by --)
with open("bank_files/full_word_graph.json", "w") as f:
    text_json = {}
    for k,v in FULL_WORD_GRAPH.items():
        # sort by weight
        v = dict(sorted(v.items(), key=lambda item: item[0]))
        text_json[k] = {f"{kk[0]}--{kk[1]}":vv for kk,vv in v.items()}
    json.dump(text_json, f, indent=3)

In [ ]:
# save to alt json (grouped by verb)
with open("bank_files/full_word_graph_alt.json", "w") as f:
    text_json = {}
    for k in FULL_WORD_GRAPH.keys():
        v = FULL_WORD_GRAPH[k]
        verbs = list(set([kk[0] for kk in v.keys()]))
        new_v = {verb: {} for verb in verbs}
        for kk,vv in v.items():
            new_v[kk[0]][kk[1]] = vv

        # sort by verb name
        new_v = dict(sorted(new_v.items(), key=lambda item: item[0]))
        text_json[k] = new_v
    json.dump(text_json, f, indent=3)

In [84]:
# save to another alt json (no weights)
with open("bank_files/full_word_graph_noweight.json", "w") as f:
    text_json = {}
    for k in FULL_WORD_GRAPH.keys():
        v = FULL_WORD_GRAPH[k]
        verbs = list(set([kk[0] for kk in v.keys()]))
        new_v = {verb: [] for verb in verbs}
        for kk,vv in v.items():
            new_v[kk[0]].append(kk[1])
        text_json[k] = new_v
    json.dump(text_json, f, indent=3)

### AF Association 

In [ ]:
# make a word dump files (noun + verb)
# will be used as an interestingness comparison of graphed words vs non-graphed words (from the random dump)

with open("bank_files/full_word_graph_noweight.json", "r") as f:
    full_graph = json.load(f)

noun_dump = set()
verb_dump = set()
for noun, rels in full_graph.items():
    noun_dump.add(noun)
    for verb, nouns in rels.items():
        verb_dump.add(verb)
        for n in nouns:
            noun_dump.add(n)

with open("bank_files/noun_dump.txt", "w") as f:
    for n in sorted(list(noun_dump)):
        f.write(f"{n}\n")

with open("bank_files/verb_dump.txt", "w") as f:
    for v in sorted(list(verb_dump)):
        f.write(f"{v}\n")


In [ ]:
# read in stupid log
with open("stupid_log.txt", "r") as f:
    STUPID_LOG = f.readlines()

AF_VERBS = ["moved", "died", "cloned", "took", "pushed", "added", "transformed", "blocked"]


AF Entities: ['[A.c921]', '[^.3522]', '[l.b6a3]', '[X.78cb]', '[_.4320]', '[q.cf80]', '[t.b36a]', '[[.3fbc]', '[`.38d2]', '[O.df00]', '[z.3203]']


In [ ]:
# search problem for best matching relationships
class AF_Story:
    def __init__(self, og_text, word_graph):
        self.og_text = og_text
        self.og_ents = self.find_entities()
        self.word_graph = word_graph

    def find_entities(self):
        ''' Find all AF entities in the original text. '''
        af_ents = []
        for line in self.og_text:
            match = re.findall(r'(\[.\..{4}\])', line)
            if match:
                af_ents.extend(match)
        return list(set(af_ents))

    